# Sesión 03 - Lab 2: Auto Loader

Este laboratorio construye un stream con Auto Loader sobre eventos de clickstream de una app de e-commerce, provoca en vivo un cambio de esquema (schema evolution) y un dato con tipo inesperado (`rescuedDataColumn`); cierra con un ejemplo de ingesta near real-time sobre un dataset distinto (sensores de un almacén). Antes de correrlo, sube `eventos_lote1.csv` y `eventos_lote2.csv` al volume `/Volumes/dbassociate/default/vol_landing/sesion_03/eventos/`. `eventos_lote3_dispositivo.csv` y `eventos_lote4_calidad.csv` se suben más adelante, en los Labs 2C y 2D; `sensores_lote1.csv` y `sensores_lote2.csv` se suben en el Lab 2F, a la carpeta `sesion_03/sensores/`.

## Verificación del entorno

In [ ]:
dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_03")


## Lab 2A: Configurar rutas de Auto Loader

El `checkpointLocation` y el `schemaLocation` van siempre bajo un Volume de Unity Catalog, nunca en `/tmp/` de un cluster efímero — el mismo criterio que ya se aplicó a los checkpoints de streaming en sesiones anteriores.

A diferencia de la lectura batch con `StructType` explícito de la Sesión 02, aquí no se provee un `schema` fijo: hacerlo forzaría el modo de evolución a `none` y desactivaría justamente el mecanismo que este lab necesita demostrar (`addNewColumns`, el modo por defecto). En su lugar, se usa `cloudFiles.schemaHints` para fijar el tipo de las columnas numéricas, sin perder la evolución automática de esquema.

In [ ]:
checkpoint_path = "/Volumes/dbassociate/default/vol_landing/sesion_03/checkpoints/eventos_lab2"
schema_path = "/Volumes/dbassociate/default/vol_landing/sesion_03/schemas/eventos_lab2"
origen_path = "/Volumes/dbassociate/default/vol_landing/sesion_03/eventos/"

print("checkpointLocation:", checkpoint_path)
print("schemaLocation:", schema_path)


## Lab 2B: Primer stream (esquema base)

`trigger(availableNow=True)` procesa todo lo disponible en `eventos/` (`eventos_lote1.csv` y `eventos_lote2.csv`) y se detiene — nunca `trigger(once=True)`, deprecado. `rescuedDataColumn` queda activo desde el primer momento, listo para el Lab 2D.

In [ ]:
from pyspark.sql.functions import current_timestamp, lit
# Importa funciones de columna: current_timestamp() genera el instante actual, lit() crea un valor constante

def construir_stream():
    return (
        spark.readStream.format("cloudFiles") # Inicia una lectura streaming usando Auto Loader (motor "cloudFiles" de Databricks)
        .option("cloudFiles.format", "csv") # Le dice a Auto Loader que los archivos de origen son CSV
        .option("cloudFiles.schemaLocation", schema_path) # Carpeta en el Volume donde Auto Loader persiste el esquema inferido/evolucionado entre corridas
        .option("cloudFiles.schemaHints", "evento_id INT, usuario_id INT, producto_id INT") # Fuerza el tipo de estas 3 columnas (si no, Auto Loader las infiere como string); el resto del esquema se sigue infiriendo solo
        .option("header", "true") # La primera fila de cada CSV trae los nombres de columna
        .option("rescuedDataColumn", "_rescued_data") # Cualquier dato que no calce con el esquema (tipo o columna inesperada) se guarda aquí en vez de romper el pipeline
        .load(origen_path) # Carpeta del Volume que Auto Loader monitorea (eventos_lote1.csv, eventos_lote2.csv, ...)
        .withColumn("ingestion_timestamp", current_timestamp()) # Columna técnica de auditoría: cuándo se ingirió cada fila
        .withColumn("source_system", lit("app_ecommerce")) # Columna técnica de auditoría: de qué sistema origen viene el dato (valor fijo)
    )

In [ ]:


query = (
    construir_stream().writeStream # Toma la definición del stream de lectura y arranca la escritura
    .option("checkpointLocation", checkpoint_path) # Dónde Auto Loader guarda el progreso (offsets) para no reprocesar ni perder datos si se reinicia
    .trigger(availableNow=True) # Procesa todo lo que haya disponible ahora mismo en lotes y se detiene solo (no queda corriendo indefinidamente) — nunca trigger(once=True), que está deprecado
    .toTable("dbassociate.default.eventos_lab2") # Tabla destino (managed) donde se van agregando (append) los datos
)
query.awaitTermination() # Bloquea la celda hasta que el stream termina de procesar el batch disponible

print("Filas cargadas en la primera corrida:", spark.table("dbassociate.default.eventos_lab2").count()) # Verificación rápida: cuántas filas quedaron en la tabla tras esta corrida

## Lab 2C: Schema evolution — sube `eventos_lote3_dispositivo.csv`

Sube `eventos_lote3_dispositivo.csv` (agrega la columna `dispositivo`) a `/Volumes/dbassociate/default/vol_landing/sesion_03/eventos/` y corré la celda siguiente. Con `cloudFiles.schemaEvolutionMode` en su valor por defecto (`addNewColumns`), Auto Loader detecta la columna nueva, actualiza `schemaLocation`, y el stream se detiene con `UnknownFieldException` — comportamiento esperado, no un error a resolver.

In [ ]:
try:
    query = (
        construir_stream().writeStream
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable("dbassociate.default.eventos_lab2")
    )
    query.awaitTermination()
except Exception as e:
    print("El stream se detuvo al detectar la columna nueva 'dispositivo' (comportamiento esperado de addNewColumns):")
    print(str(e)[:400])


Reiniciar el stream retoma el procesamiento con el esquema ya actualizado. Las filas cargadas antes del cambio quedan con `dispositivo = NULL`; las del lote 3 en adelante, con el valor real.

Auto Loader evoluciona su propio `schemaLocation`, pero eso no evoluciona automáticamente el esquema de la tabla Delta destino — son dos esquemas persistidos por separado. Con Table ACLs habilitado (el caso por defecto bajo Unity Catalog), la migración automática queda bloqueada y la escritura falla con `DELTA_METADATA_MISMATCH` si no se agrega explícitamente `.option("mergeSchema", "true")` al `writeStream`, como ya está en la celda siguiente.

In [ ]:
query = (
    construir_stream().writeStream
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("dbassociate.default.eventos_lab2")
)
query.awaitTermination()

spark.sql("""
    SELECT dispositivo, COUNT(*) AS eventos
    FROM dbassociate.default.eventos_lab2
    GROUP BY dispositivo
""").show(truncate=False)


## Lab 2D: rescuedDataColumn — sube `eventos_lote4_calidad.csv`

`eventos_lote4_calidad.csv` trae dos filas con `usuario_id = "invitado"` en vez de un número, un mismatch de tipo contra el schema hint `usuario_id INT`. Sube el archivo a la misma carpeta y corré la celda: el pipeline no se rompe, esas filas quedan con `usuario_id` nulo y el valor original capturado en `_rescued_data`.

In [ ]:
query = (
    construir_stream().writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("dbassociate.default.eventos_lab2")
)
query.awaitTermination()

spark.sql("""
    SELECT evento_id, usuario_id, _rescued_data
    FROM dbassociate.default.eventos_lab2
    WHERE _rescued_data IS NOT NULL
""").show(truncate=False)


## Lab 2E: Validación final y progreso del stream

Consulta agregada de cierre, más una lectura rápida de `recentProgress` de la última corrida — la forma más directa de observar cuántas filas procesó cada micro-batch, sin salir del notebook.

In [ ]:
spark.sql("""
    SELECT tipo_evento, dispositivo, COUNT(*) AS total_eventos
    FROM dbassociate.default.eventos_lab2
    GROUP BY tipo_evento, dispositivo
    ORDER BY tipo_evento, dispositivo
""").show(truncate=False)

print("Total de filas en la tabla:", spark.table("dbassociate.default.eventos_lab2").count())

print("Progreso de la última corrida (filas procesadas por micro-batch):")
for batch in query.recentProgress:
    print(batch.get("batchId"), "->", batch.get("numInputRows"), "filas")


## Lab 2F: Ingesta near real-time con `trigger(processingTime)`

Los Labs 2B-2E usan `trigger(availableNow=True)`: procesa todo lo disponible y se detiene solo — el patrón más económico, pensado para cargas programadas (ver Sílabo, Sesión 3: "Tuning de triggers y tamaño de micro-batches"). Este lab usa `trigger(processingTime="15 seconds")`: el stream queda activo y revisa la carpeta de origen cada 15 segundos, sin que haya que relanzar la celda para recoger archivos nuevos — el patrón para dashboards o alertas casi en tiempo real. Dataset distinto a los anteriores: lecturas de sensores de temperatura/humedad de un almacén. Antes de correr la siguiente celda, sube `sensores_lote1.csv` a `/Volumes/dbassociate/default/vol_landing/sesion_03/sensores/` (`sensores_lote2.csv` se sube más adelante, con el stream ya corriendo).

In [ ]:
checkpoint_path_sensores = "/Volumes/dbassociate/default/vol_landing/sesion_03/checkpoints/sensores_lab2"
schema_path_sensores = "/Volumes/dbassociate/default/vol_landing/sesion_03/schemas/sensores_lab2"
origen_path_sensores = "/Volumes/dbassociate/default/vol_landing/sesion_03/sensores/"

print("checkpointLocation:", checkpoint_path_sensores)
print("schemaLocation:", schema_path_sensores)


In [ ]:
def construir_stream_sensores():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path_sensores)
        .option("cloudFiles.schemaHints", "sensor_id INT, temperatura DOUBLE, humedad_relativa DOUBLE")
        .option("header", "true")
        .option("rescuedDataColumn", "_rescued_data")
        .load(origen_path_sensores)
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_system", lit("sensores_almacen"))
    )


In [ ]:
query_sensores = (
    construir_stream_sensores().writeStream
    .option("checkpointLocation", checkpoint_path_sensores)
    .trigger(processingTime="15 seconds")
    .toTable("dbassociate.default.sensores_lab2")
)

print("Stream near real-time iniciado. isActive:", query_sensores.isActive)
print("Revisa la carpeta 'sensores/' cada 15 segundos, sin detenerse. No corras query_sensores.awaitTermination() todavía: pasa a la siguiente celda.")


El stream de la celda anterior ya quedó corriendo en segundo plano — a diferencia de `availableNow=True`, esta celda termina de ejecutarse sin que el stream se detenga. Sube ahora `sensores_lote2.csv` a la misma carpeta `sensores/`, sin tocar nada más, y espera unos 20-30 segundos (más de un ciclo de 15 segundos) antes de correr la siguiente celda.

In [ ]:
print("Filas ingeridas hasta ahora:", spark.table("dbassociate.default.sensores_lab2").count())

spark.sql("""
    SELECT sensor_id, ubicacion, temperatura, humedad_relativa, timestamp_lectura
    FROM dbassociate.default.sensores_lab2
    ORDER BY ingestion_timestamp DESC
    LIMIT 15
""").show(truncate=False)

print("Progreso por micro-batch (algunos con 0 filas son ciclos de polling sin archivos nuevos, comportamiento esperado):")
for batch in query_sensores.recentProgress:
    print(batch.get("batchId"), "->", batch.get("numInputRows"), "filas")


Con `processingTime`, el stream sigue consumiendo cómputo indefinidamente hasta que se detiene explícitamente — a diferencia de `availableNow=True`, que se autotermina. Nunca dejar un stream de este tipo corriendo sin supervisión en un lab. Para latencias por debajo del segundo existe además `trigger(realTime=True)` (Real-Time Mode), pero requiere Photon y un cluster de tamaño fijo — fuera del alcance de este curso, que corre sobre compute serverless.

In [ ]:
query_sensores.stop()
print("Stream near real-time detenido. isActive:", query_sensores.isActive)


## Limpieza

In [ ]:
spark.sql("DROP TABLE IF EXISTS dbassociate.default.eventos_lab2")
dbutils.fs.rm(checkpoint_path, recurse=True)
dbutils.fs.rm(schema_path, recurse=True)

spark.sql("DROP TABLE IF EXISTS dbassociate.default.sensores_lab2")
dbutils.fs.rm(checkpoint_path_sensores, recurse=True)
dbutils.fs.rm(schema_path_sensores, recurse=True)

print("Tablas y estado de streaming de este laboratorio eliminados.")
